In [1]:
!nvidia-smi

Thu Aug 13 15:41:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q sentence-transformers datasets accelerate
!pip install -q faiss-cpu pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 97.2 MB/s eta 0:00:00


In [3]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

GPU available: True
Tesla T4


In [5]:
query = "query: What does the Gita say about karma?"

passage = """
passage: कर्मण्येवाधिकारस्ते मा फलेषु कदाचन
कर्मण्येवाधिकारस्ते मा फलेषु कदाचन ।
You have a right to perform your prescribed duties, but not to the fruits of your actions.
"""

In [8]:
#install libraries
!pip install pandas numpy scikit-learn sentence-transformers torch

# DATASET CREATION AND PREPARATION

In [9]:
from datasets import load_dataset
import pandas as pd

# Load dataset
ds = load_dataset("JDhruv14/Bhagavad-Gita_Dataset")

# Convert train split to pandas
df = ds["train"].to_pandas()

print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())

# 1. Keep only required columns

df = df[
    [
        "chapter",
        "verse",
        "sanskrit",
        "english",
        "transliteration"
    ]
].copy()

# 2. Remove rows with missing Sanskrit/English

df = df.dropna(
    subset=["sanskrit", "english"]
)

# 3. Convert text columns to string

for col in ["sanskrit", "english", "transliteration"]:
    df[col] = df[col].astype(str)


# 4. Clean whitespace

for col in ["sanskrit", "english", "transliteration"]:

    df[col] = (
        df[col]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# 5. Create unique verse ID

df["verse_id"] = (
    "BG_"
    + df["chapter"].astype(str)
    + "_"
    + df["verse"].astype(str)
)

# 6. Remove duplicate verses

df = df.drop_duplicates(
    subset=["verse_id"]
)

# 7. Sort by chapter and verse

df = df.sort_values(
    ["chapter", "verse"]
).reset_index(drop=True)

# 8. Check number of verses

print("Clean verses:", len(df))

print("\nFirst 5 rows:")
print(
    df[
        [
            "verse_id",
            "chapter",
            "verse",
            "sanskrit",
            "english"
        ]
    ].head()
)

README.md:   0%|          | 0.00/952 [00:00<?, ?B/s]

geeta_dataset.csv:   0%|          | 0.00/677k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/701 [00:00<?, ? examples/s]

Original shape: (701, 6)
Columns: ['chapter', 'verse', 'sanskrit', 'hindi', 'english', 'transliteration']
Clean verses: 701

First 5 rows:
  verse_id  chapter  verse                                           sanskrit  \
0   BG_1_1        1      1  धृतराष्ट्र उवाच |धर्मक्षेत्रे कुरुक्षेत्रे समव...   
1   BG_1_2        1      2  सञ्जय उवाच |दृष्ट्वा तु पाण्डवानीकं व्यूढं दुर...   
2   BG_1_3        1      3  पश्यैतां पाण्डुपुत्राणामाचार्य महतीं चमूम् |व्...   
3   BG_1_4        1      4  अत्र शूरा महेष्वासा भीमार्जुनसमा युधि |युयुधान...   
4   BG_1_5        1      5  धृष्टकेतुश्चेकितानः काशिराजश्च वीर्यवान् |पुरु...   

                                             english  
0  Dhritarashtra said: Sanjaya, gathered on the s...  
1  Sanjaya said: At that time, seeing the army of...  
2  Behold, Master, the mighty army of the sons of...  
3  The strong Yodhamanyu and the brave Uttamaujas...  
4  Dhrishtaketu, chekitana and the valiant king o...  


In [10]:
print("Total verses:", len(df))

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate verse IDs:")
print(df["verse_id"].duplicated().sum())

print("\nChapters:")
print(df["chapter"].unique())

print("\nVerses per chapter:")
print(df.groupby("chapter")["verse"].count())

Total verses: 701

Missing values:
chapter            0
verse              0
sanskrit           0
english            0
transliteration    0
verse_id           0
dtype: int64

Duplicate verse IDs:
0

Chapters:
[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]

Verses per chapter:
chapter
1     47
2     72
3     43
4     42
5     29
6     47
7     30
8     28
9     34
10    42
11    55
12    20
13    35
14    27
15    20
16    24
17    28
18    78
Name: verse, dtype: int64


In [11]:
pd.set_option("display.max_colwidth", 300)

print(
    df[
        [
            "verse_id",
            "sanskrit",
            "english",
            "transliteration"
        ]
    ].sample(10, random_state=42)
)

     verse_id  \
680  BG_18_58   
164    BG_4_3   
54     BG_2_8   
640  BG_18_18   
606  BG_17_12   
399  BG_10_28   
575   BG_16_5   
668  BG_18_46   
332   BG_8_23   
363   BG_9_26   

                                                                                                           sanskrit  \
680                      मच्चित्तः सर्वदुर्गाणि मत्प्रसादात्तरिष्यसि |अथ चेत्त्वमहंकारान्न श्रोष्यसि विनङ्क्ष्यसि |   
164                           स एवायं मया तेऽद्य योगः प्रोक्तः पुरातनः |भक्तोऽसि मे सखा चेति रहस्यं ह्येतदुत्तमम् |   
54   न हि प्रपश्यामि ममापनुद्याद्यच्छोकमुच्छोषणमिन्द्रियाणाम् |अवाप्य भूमावसपत्नमृद्धंराज्यं सुराणामपि चाधिपत्यम् |   
640                            ज्ञानं ज्ञेयं परिज्ञाता त्रिविधा कर्मचोदना |करणं कर्म कर्तेति त्रिविधः कर्मसंग्रहः |   
606                                अभिसन्धाय तु फलं दम्भार्थमपि चैव यत् |इज्यते भरतश्रेष्ठ तं यज्ञं विद्धि राजसम् |   
399                             आयुधानामहं वज्रं धेनूनामस्मि कामधुक् |प्रजनश्चास्मि कन्दर्पः सर्पा

In [12]:
df.to_csv(
    "gita_clean_master.csv",
    index=False,
    encoding="utf-8"
)

print("Saved: gita_clean_master.csv")

Saved: gita_clean_master.csv


In [19]:
print("Total verses:", len(df))
print(df.isnull().sum())
print(df.head())

Total verses: 701
chapter            0
verse              0
sanskrit           0
english            0
transliteration    0
verse_id           0
dtype: int64
   chapter  verse  \
0        1      1   
1        1      2   
2        1      3   
3        1      4   
4        1      5   

                                                                                             sanskrit  \
0  धृतराष्ट्र उवाच |धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः |मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय |   
1        सञ्जय उवाच |दृष्ट्वा तु पाण्डवानीकं व्यूढं दुर्योधनस्तदा |आचार्यमुपसंगम्य राजा वचनमब्रवीत् |   
2                पश्यैतां पाण्डुपुत्राणामाचार्य महतीं चमूम् |व्यूढां द्रुपदपुत्रेण तव शिष्येण धीमता |   
3                         अत्र शूरा महेष्वासा भीमार्जुनसमा युधि |युयुधानो विराटश्च द्रुपदश्च महारथः |   
4                  धृष्टकेतुश्चेकितानः काशिराजश्च वीर्यवान् |पुरुजित्कुन्तिभोजश्च शैब्यश्च नरपुंगवः |   

                                                                                     

In [20]:
# Check duplicate chapter + verse combinations
duplicates = df[df.duplicated(
    subset=["chapter", "verse"],
    keep=False
)]

print("Duplicate chapter-verse rows:")
print(duplicates[["chapter", "verse", "verse_id"]])

print("\nNumber of duplicate rows:", len(duplicates))

Duplicate chapter-verse rows:
Empty DataFrame
Columns: [chapter, verse, verse_id]
Index: []

Number of duplicate rows: 0


In [21]:
print("\nVerses per chapter:")
print(
    df.groupby("chapter")["verse"]
      .count()
      .to_string()
)
print("\nMaximum verse number in each chapter:")
print(
    df.groupby("chapter")["verse"]
      .max()
      .to_string()
)


Verses per chapter:
chapter
1     47
2     72
3     43
4     42
5     29
6     47
7     30
8     28
9     34
10    42
11    55
12    20
13    35
14    27
15    20
16    24
17    28
18    78

Maximum verse number in each chapter:
chapter
1     47
2     72
3     43
4     42
5     29
6     47
7     30
8     28
9     34
10    42
11    55
12    20
13    35
14    27
15    20
16    24
17    28
18    78


In [22]:
chapter_counts = (
    df.groupby("chapter")["verse"]
      .count()
)

print(chapter_counts)
print("\nTotal:", chapter_counts.sum())

chapter
1     47
2     72
3     43
4     42
5     29
6     47
7     30
8     28
9     34
10    42
11    55
12    20
13    35
14    27
15    20
16    24
17    28
18    78
Name: verse, dtype: int64

Total: 701


In [23]:
# Sort all 701 verses
df = df.sort_values(
    ["chapter", "verse"]
).reset_index(drop=True)

print("Total verses:", len(df))

# Save complete cleaned dataset
df.to_csv(
    "gita_701_clean.csv",
    index=False,
    encoding="utf-8"
)

Total verses: 701


# TRAIN-TEST SPLIT

In [24]:
from sklearn.model_selection import train_test_split
# First split: 80% train, 20% temporary

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["chapter"],
    random_state=42
)

# Second split: temporary → validation/test

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["chapter"],
    random_state=42
)

# Sort for readability
train_df = train_df.sort_values(
    ["chapter", "verse"]
).reset_index(drop=True)

val_df = val_df.sort_values(
    ["chapter", "verse"]
).reset_index(drop=True)

test_df = test_df.sort_values(
    ["chapter", "verse"]
).reset_index(drop=True)

# Check sizes
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Train: 560
Validation: 70
Test: 71
Total: 701


In [25]:
print("\nTrain distribution:")
print(train_df.groupby("chapter").size())

print("\nValidation distribution:")
print(val_df.groupby("chapter").size())

print("\nTest distribution:")
print(test_df.groupby("chapter").size())


Train distribution:
chapter
1     38
2     57
3     34
4     34
5     23
6     38
7     24
8     22
9     27
10    34
11    44
12    16
13    28
14    22
15    16
16    19
17    22
18    62
dtype: int64

Validation distribution:
chapter
1     4
2     7
3     4
4     4
5     3
6     4
7     3
8     3
9     4
10    4
11    5
12    2
13    4
14    3
15    2
16    3
17    3
18    8
dtype: int64

Test distribution:
chapter
1     5
2     8
3     5
4     4
5     3
6     5
7     3
8     3
9     3
10    4
11    6
12    2
13    3
14    2
15    2
16    2
17    3
18    8
dtype: int64


In [26]:
train_ids = set(train_df["verse_id"])
val_ids = set(val_df["verse_id"])
test_ids = set(test_df["verse_id"])

print(
    "Train ∩ Validation:",
    len(train_ids & val_ids)
)

print(
    "Train ∩ Test:",
    len(train_ids & test_ids)
)

print(
    "Validation ∩ Test:",
    len(val_ids & test_ids)
)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [27]:
train_df.to_csv(
    "train_documents.csv",
    index=False,
    encoding="utf-8"
)

val_df.to_csv(
    "val_documents.csv",
    index=False,
    encoding="utf-8"
)

test_df.to_csv(
    "test_documents.csv",
    index=False,
    encoding="utf-8"
)

print("Train, validation and test files saved.")

Train, validation and test files saved.


In [28]:
positive_pairs = []

for _, row in train_df.iterrows():

    # -----------------------------------------
    # Sanskrit → English
    # -----------------------------------------

    positive_pairs.append({
        "query_id": f'{row["verse_id"]}_sa_en',
        "verse_id": row["verse_id"],
        "query": row["sanskrit"],
        "positive": row["english"],
        "query_language": "sanskrit",
        "positive_language": "english"
    })

    # -----------------------------------------
    # English → Sanskrit
    # -----------------------------------------

    positive_pairs.append({
        "query_id": f'{row["verse_id"]}_en_sa',
        "verse_id": row["verse_id"],
        "query": row["english"],
        "positive": row["sanskrit"],
        "query_language": "english",
        "positive_language": "sanskrit"
    })


pairs_df = pd.DataFrame(positive_pairs)

print("Number of positive pairs:", len(pairs_df))

print("\nFirst 5 pairs:")
print(pairs_df.head())

Number of positive pairs: 1120

First 5 pairs:
       query_id verse_id  \
0  BG_1_1_sa_en   BG_1_1   
1  BG_1_1_en_sa   BG_1_1   
2  BG_1_2_sa_en   BG_1_2   
3  BG_1_2_en_sa   BG_1_2   
4  BG_1_3_sa_en   BG_1_3   

                                                                                                                                            query  \
0                                              धृतराष्ट्र उवाच |धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः |मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय |   
1     Dhritarashtra said: Sanjaya, gathered on the sacred soil of Kurukshetra, eager to fight, what did my children and the children of Pandu do?   
2                                                    सञ्जय उवाच |दृष्ट्वा तु पाण्डवानीकं व्यूढं दुर्योधनस्तदा |आचार्यमुपसंगम्य राजा वचनमब्रवीत् |   
3  Sanjaya said: At that time, seeing the army of the Pandavas drawn up for battle and approaching Dronacharya King Duryodhana spoke these words:   
4                                      

In [29]:
pairs_df.to_csv(
    "train_positive_pairs.csv",
    index=False,
    encoding="utf-8"
)

print("Saved train_positive_pairs.csv")

Saved train_positive_pairs.csv


In [30]:
pd.set_option("display.max_colwidth", 300)

print(
    pairs_df.sample(
        10,
        random_state=42
    )[
        [
            "query_id",
            "verse_id",
            "query",
            "positive",
            "query_language",
            "positive_language"
        ]
    ].to_string(index=False)
)

      query_id verse_id                                                                                                                                                                                           query                                                                                                                                                                         positive query_language positive_language
 BG_3_31_en_sa  BG_3_31                                                      Even those men who, with an uncavilling and devout mind, always follow this teaching of Mine are released from the bondage of all actions.                                                                                           ये मे मतमिदं नित्यमनुतिष्ठन्ति मानवाः |श्रद्धावन्तोऽनसूयन्तो मुच्यन्ते तेऽपि कर्मभिः |        english          sanskrit
 BG_2_19_en_sa  BG_2_19                                                            All these bodies pertaining to the imperishable, indefinable and 

# LOAD MODEL

In [31]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "intfloat/multilingual-e5-small"
)

print("E5 model loaded.")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

E5 model loaded.


In [32]:
# English documents
english_documents = train_df[
    ["verse_id", "english"]
].copy()

# Sanskrit documents
sanskrit_documents = train_df[
    ["verse_id", "sanskrit"]
].copy()

print("English documents:", len(english_documents))
print("Sanskrit documents:", len(sanskrit_documents))

English documents: 560
Sanskrit documents: 560


In [33]:
english_texts = [
    "passage: " + text
    for text in english_documents["english"]
]

english_embeddings = model.encode(
    english_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("English embedding shape:", english_embeddings.shape)

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

English embedding shape: (560, 384)


In [34]:
sanskrit_texts = [
    "passage: " + text
    for text in sanskrit_documents["sanskrit"]
]

sanskrit_embeddings = model.encode(
    sanskrit_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(
    "Sanskrit embedding shape:",
    sanskrit_embeddings.shape
)

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

Sanskrit embedding shape: (560, 384)


In [35]:
sa_en_pairs = pairs_df[
    pairs_df["query_language"] == "sanskrit"
].reset_index(drop=True)

print("Sanskrit → English queries:", len(sa_en_pairs))

Sanskrit → English queries: 560


In [36]:
sa_queries = [
    "query: " + text
    for text in sa_en_pairs["query"]
]

sa_query_embeddings = model.encode(
    sa_queries,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

In [37]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sa_similarity = cosine_similarity(
    sa_query_embeddings,
    english_embeddings
)

print(sa_similarity.shape)

(560, 560)


In [38]:
sa_hard_negatives = []

for i, row in sa_en_pairs.iterrows():

    scores = sa_similarity[i]

    # Rank documents by similarity
    ranked_indices = np.argsort(scores)[::-1]

    positive_id = row["verse_id"]

    # Find highest-ranked incorrect document
    negative_id = None

    for idx in ranked_indices:

        candidate_id = english_documents.iloc[idx]["verse_id"]

        if candidate_id != positive_id:
            negative_id = candidate_id
            break

    negative_row = english_documents[
        english_documents["verse_id"] == negative_id
    ].iloc[0]

    sa_hard_negatives.append({
        "query_id": row["query_id"],
        "query": row["query"],
        "positive": row["positive"],
        "negative": negative_row["english"],
        "positive_id": positive_id,
        "negative_id": negative_id,
        "query_language": "sanskrit"
    })

sa_hard_df = pd.DataFrame(sa_hard_negatives)

print(sa_hard_df.head())

       query_id  \
0  BG_1_1_sa_en   
1  BG_1_2_sa_en   
2  BG_1_3_sa_en   
3  BG_1_4_sa_en   
4  BG_1_5_sa_en   

                                                                                                query  \
0  धृतराष्ट्र उवाच |धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः |मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय |   
1        सञ्जय उवाच |दृष्ट्वा तु पाण्डवानीकं व्यूढं दुर्योधनस्तदा |आचार्यमुपसंगम्य राजा वचनमब्रवीत् |   
2                पश्यैतां पाण्डुपुत्राणामाचार्य महतीं चमूम् |व्यूढां द्रुपदपुत्रेण तव शिष्येण धीमता |   
3                         अत्र शूरा महेष्वासा भीमार्जुनसमा युधि |युयुधानो विराटश्च द्रुपदश्च महारथः |   
4                  धृष्टकेतुश्चेकितानः काशिराजश्च वीर्यवान् |पुरुजित्कुन्तिभोजश्च शैब्यश्च नरपुंगवः |   

                                                                                                                                         positive  \
0     Dhritarashtra said: Sanjaya, gathered on the sacred soil of Kurukshetra, eager to fight, what did m

In [39]:
en_sa_pairs = pairs_df[
    pairs_df["query_language"] == "english"
].reset_index(drop=True)

print("English → Sanskrit queries:", len(en_sa_pairs))

English → Sanskrit queries: 560


In [40]:
en_queries = [
    "query: " + text
    for text in en_sa_pairs["query"]
]

en_query_embeddings = model.encode(
    en_queries,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

In [41]:
en_similarity = cosine_similarity(
    en_query_embeddings,
    sanskrit_embeddings
)

In [42]:
en_hard_negatives = []

for i, row in en_sa_pairs.iterrows():

    scores = en_similarity[i]

    ranked_indices = np.argsort(scores)[::-1]

    positive_id = row["verse_id"]

    negative_id = None

    for idx in ranked_indices:

        candidate_id = sanskrit_documents.iloc[idx]["verse_id"]

        if candidate_id != positive_id:
            negative_id = candidate_id
            break

    negative_row = sanskrit_documents[
        sanskrit_documents["verse_id"] == negative_id
    ].iloc[0]

    en_hard_negatives.append({
        "query_id": row["query_id"],
        "query": row["query"],
        "positive": row["positive"],
        "negative": negative_row["sanskrit"],
        "positive_id": positive_id,
        "negative_id": negative_id,
        "query_language": "english"
    })

en_hard_df = pd.DataFrame(en_hard_negatives)

print(en_hard_df.head())

       query_id  \
0  BG_1_1_en_sa   
1  BG_1_2_en_sa   
2  BG_1_3_en_sa   
3  BG_1_4_en_sa   
4  BG_1_5_en_sa   

                                                                                                                                            query  \
0     Dhritarashtra said: Sanjaya, gathered on the sacred soil of Kurukshetra, eager to fight, what did my children and the children of Pandu do?   
1  Sanjaya said: At that time, seeing the army of the Pandavas drawn up for battle and approaching Dronacharya King Duryodhana spoke these words:   
2                  Behold, Master, the mighty army of the sons of Pandu arrayed for battle by your talented pupil, Dhristadyumna, son of Drupada.   
3                           The strong Yodhamanyu and the brave Uttamaujas, the son\nof Subhadra, and the sons of Draupadi, all of great chariots   
4                                           Dhrishtaketu, chekitana and the valiant king of Kasi, Purujit and Kuntibhoja and Saibya, the bes

In [43]:
hard_negative_df = pd.concat(
    [
        sa_hard_df,
        en_hard_df
    ],
    ignore_index=True
)

print(
    "Total hard-negative candidates:",
    len(hard_negative_df)
)

hard_negative_df.to_csv(
    "hard_negative_candidates.csv",
    index=False,
    encoding="utf-8"
)

Total hard-negative candidates: 1120


In [44]:
import pandas as pd

# Load E5 hard-negative candidates
hard_negative_df = pd.read_csv(
    "hard_negative_candidates.csv"
)

# Create triplets
triplets_df = hard_negative_df[
    [
        "query",
        "positive",
        "negative"
    ]
].copy()

print("Number of triplets:", len(triplets_df))

print("\nSample triplets:")
print(triplets_df.head())

Number of triplets: 1120

Sample triplets:
                                                                                                query  \
0  धृतराष्ट्र उवाच |धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः |मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय |   
1        सञ्जय उवाच |दृष्ट्वा तु पाण्डवानीकं व्यूढं दुर्योधनस्तदा |आचार्यमुपसंगम्य राजा वचनमब्रवीत् |   
2                पश्यैतां पाण्डुपुत्राणामाचार्य महतीं चमूम् |व्यूढां द्रुपदपुत्रेण तव शिष्येण धीमता |   
3                         अत्र शूरा महेष्वासा भीमार्जुनसमा युधि |युयुधानो विराटश्च द्रुपदश्च महारथः |   
4                  धृष्टकेतुश्चेकितानः काशिराजश्च वीर्यवान् |पुरुजित्कुन्तिभोजश्च शैब्यश्च नरपुंगवः |   

                                                                                                                                         positive  \
0     Dhritarashtra said: Sanjaya, gathered on the sacred soil of Kurukshetra, eager to fight, what did my children and the children of Pandu do?   
1  Sanjaya said: At that tim

In [45]:
triplets_df.to_csv(
    "train_triplets.csv",
    index=False,
    encoding="utf-8"
)

print("Saved: train_triplets.csv")

Saved: train_triplets.csv


In [85]:
print("Missing values:")
print(triplets_df.isnull().sum())

print("\nDuplicate triplets:")
print(triplets_df.duplicated().sum())

print("\nTriplet dataset shape:")
print(triplets_df.shape)

Missing values:
query       0
positive    0
negative    0
dtype: int64

Duplicate triplets:
0

Triplet dataset shape:
(1120, 3)


In [46]:
val_pairs = []

for _, row in val_df.iterrows():

    # Sanskrit → English
    val_pairs.append({
        "query_id": f'{row["verse_id"]}_sa_en',
        "verse_id": row["verse_id"],
        "query": row["sanskrit"],
        "positive": row["english"],
        "query_language": "sanskrit",
        "document_language": "english"
    })

    # English → Sanskrit
    val_pairs.append({
        "query_id": f'{row["verse_id"]}_en_sa',
        "verse_id": row["verse_id"],
        "query": row["english"],
        "positive": row["sanskrit"],
        "query_language": "english",
        "document_language": "sanskrit"
    })

val_pairs_df = pd.DataFrame(val_pairs)

print("Validation pairs:", len(val_pairs_df))

Validation pairs: 140


In [47]:
val_pairs_df.to_csv(
    "validation_pairs.csv",
    index=False,
    encoding="utf-8"
)

In [48]:
test_pairs = []

for _, row in test_df.iterrows():

    # Sanskrit → English
    test_pairs.append({
        "query_id": f'{row["verse_id"]}_sa_en',
        "verse_id": row["verse_id"],
        "query": row["sanskrit"],
        "positive": row["english"],
        "query_language": "sanskrit",
        "document_language": "english"
    })

    # English → Sanskrit
    test_pairs.append({
        "query_id": f'{row["verse_id"]}_en_sa',
        "verse_id": row["verse_id"],
        "query": row["english"],
        "positive": row["sanskrit"],
        "query_language": "english",
        "document_language": "sanskrit"
    })

test_pairs_df = pd.DataFrame(test_pairs)

print("Test pairs:", len(test_pairs_df))

Test pairs: 142


In [49]:
test_pairs_df.to_csv(
    "test_pairs.csv",
    index=False,
    encoding="utf-8"
)

In [50]:
pip install pandas numpy torch sentence-transformers scikit-learn

In [51]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("train_documents.csv")
val_df = pd.read_csv("val_documents.csv")
test_df = pd.read_csv("test_documents.csv")

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 560
Validation: 70
Test: 71


In [52]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "intfloat/multilingual-e5-small"
)

print("Model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded.


In [53]:
test_queries = []

for _, row in test_df.iterrows():

    test_queries.append({
        "query_id": f'{row["verse_id"]}_sa_en',
        "query": row["sanskrit"],
        "target_id": row["verse_id"],
        "direction": "sa_en"
    })

    test_queries.append({
        "query_id": f'{row["verse_id"]}_en_sa',
        "query": row["english"],
        "target_id": row["verse_id"],
        "direction": "en_sa"
    })

test_queries_df = pd.DataFrame(test_queries)

print(len(test_queries_df))

142


In [54]:
def retrieve(
    model,
    queries,
    documents,
    document_embeddings,
    k=10
):

    query_embeddings = model.encode(
        [
            "query: " + q
            for q in queries
        ],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embeddings,
        document_embeddings
    )

    rankings = np.argsort(
        scores,
        axis=1
    )[:, ::-1]

    return rankings, scores

In [55]:
test_sa_en = test_queries_df[
    test_queries_df["direction"] == "sa_en"
].reset_index(drop=True)

test_sa_queries = test_sa_en["query"].tolist()

# Documents = English test verses
test_english_docs = test_df["english"].tolist()

test_english_embeddings = model.encode(
    [
        "passage: " + text
        for text in test_english_docs
    ],
    normalize_embeddings=True
)

sa_rankings, sa_scores = retrieve(
    model,
    test_sa_queries,
    test_english_docs,
    test_english_embeddings,
    k=10
)

In [56]:
def calculate_metrics(
    rankings,
    target_ids,
    document_ids,
    k_values=[1, 5, 10]
):

    results = {}

    reciprocal_ranks = []
    ndcg_scores = []

    for i, ranking in enumerate(rankings):

        target = target_ids[i]

        ranked_ids = [
            document_ids[idx]
            for idx in ranking
        ]

        # MRR
        if target in ranked_ids:
            rank = ranked_ids.index(target) + 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)

        # nDCG for one relevant document
        if target in ranked_ids:
            rank = ranked_ids.index(target) + 1
            ndcg_scores.append(
                1 / np.log2(rank + 1)
            )
        else:
            ndcg_scores.append(0)

    for k in k_values:

        hits = 0

        for i, ranking in enumerate(rankings):

            top_k_ids = [
                document_ids[idx]
                for idx in ranking[:k]
            ]

            if target_ids[i] in top_k_ids:
                hits += 1

        results[f"Recall@{k}"] = (
            hits / len(target_ids)
        )

    results["MRR"] = np.mean(
        reciprocal_ranks
    )

    results["nDCG"] = np.mean(
        ndcg_scores
    )

    return results

# TRAINED ON BASE MODEL

In [57]:
test_doc_ids = test_df["verse_id"].tolist()
test_target_ids = test_sa_en["target_id"].tolist()

baseline_sa_en = calculate_metrics(
    sa_rankings,
    test_target_ids,
    test_doc_ids
)

print(baseline_sa_en)

{'Recall@1': 0.323943661971831, 'Recall@5': 0.5774647887323944, 'Recall@10': 0.7464788732394366, 'MRR': np.float64(0.4476294058084417), 'nDCG': np.float64(0.566177751513024)}


In [58]:
test_en_sa = test_queries_df[
    test_queries_df["direction"] == "en_sa"
].reset_index(drop=True)

test_en_queries = test_en_sa["query"].tolist()

test_sanskrit_docs = test_df["sanskrit"].tolist()

test_sanskrit_embeddings = model.encode(
    [
        "passage: " + text
        for text in test_sanskrit_docs
    ],
    normalize_embeddings=True
)

en_rankings, en_scores = retrieve(
    model,
    test_en_queries,
    test_sanskrit_docs,
    test_sanskrit_embeddings,
    k=10
)

baseline_en_sa = calculate_metrics(
    en_rankings,
    test_en_sa["target_id"].tolist(),
    test_doc_ids
)

print(baseline_en_sa)

{'Recall@1': 0.7887323943661971, 'Recall@5': 0.9295774647887324, 'Recall@10': 0.9436619718309859, 'MRR': np.float64(0.847650295713676), 'nDCG': np.float64(0.8828310314408201)}


In [59]:
from sentence_transformers import SentenceTransformer, InputExample
from torch.utils.data import DataLoader
from sentence_transformers.losses import MultipleNegativesRankingLoss

/tmp/ipykernel_641/948346824.py:3: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss


In [60]:
train_examples = []

for _, row in triplets_df.iterrows():

    train_examples.append(
        InputExample(
            texts=[
                row["query"],
                row["positive"],
                row["negative"]
            ]
        )
    )

print("Training examples:", len(train_examples))

Training examples: 1120


In [64]:
train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=16
)

# FINETUNED MODEL

In [65]:
fine_tuned_model = SentenceTransformer(
    "intfloat/multilingual-e5-small"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [66]:
loss = MultipleNegativesRankingLoss(
    fine_tuned_model
)

In [67]:
fine_tuned_model.fit(
    train_objectives=[
        (train_dataloader, loss)
    ],
    epochs=3,
    warmup_steps=100,
    output_path="models/gita_e5_finetuned"
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [68]:
fine_tuned_model = SentenceTransformer(
    "models/gita_e5_finetuned"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [69]:
ft_sa_rankings, ft_sa_scores = retrieve(
    fine_tuned_model,
    test_sa_queries,
    test_english_docs,
    test_english_embeddings,
    k=10
)

In [70]:
ft_english_embeddings = fine_tuned_model.encode(
    [
        "passage: " + text
        for text in test_english_docs
    ],
    normalize_embeddings=True
)

ft_sa_rankings, ft_sa_scores = retrieve(
    fine_tuned_model,
    test_sa_queries,
    test_english_docs,
    ft_english_embeddings,
    k=10
)

# METRICS FOR FINE TUNED MODEL

In [72]:
ft_sa_metrics = calculate_metrics(
    ft_sa_rankings,
    test_sa_en["target_id"].tolist(),
    test_doc_ids
)

print("Fine-tuned Sanskrit → English")
print(ft_sa_metrics)

Fine-tuned Sanskrit → English
{'Recall@1': 0.8873239436619719, 'Recall@5': 0.9859154929577465, 'Recall@10': 0.9859154929577465, 'MRR': np.float64(0.9295774647887324), 'nDCG': np.float64(0.9468258271378387)}


In [74]:
ft_sanskrit_embeddings = fine_tuned_model.encode(
    [
        "passage: " + text
        for text in test_sanskrit_docs
    ],
    normalize_embeddings=True
)

ft_en_rankings, ft_en_scores = retrieve(
    fine_tuned_model,
    test_en_queries,
    test_sanskrit_docs,
    ft_sanskrit_embeddings,
    k=10
)

ft_en_metrics = calculate_metrics(
    ft_en_rankings,
    test_en_sa["target_id"].tolist(),
    test_doc_ids
)

print("Fine-tuned English → Sanskrit")
print(ft_en_metrics)

Fine-tuned English → Sanskrit
{'Recall@1': 0.8873239436619719, 'Recall@5': 0.9859154929577465, 'Recall@10': 1.0, 'MRR': np.float64(0.9330985915492958), 'nDCG': np.float64(0.9498807396202635)}


# COMPARISON BETWEEN BASE AND FINE TUNED MODEL

In [75]:
comparison = pd.DataFrame([
    {
        "Model": "Base E5",
        **baseline_sa_en
    },
    {
        "Model": "Fine-tuned E5",
        **ft_sa_metrics
    }
])

print(comparison)

           Model  Recall@1  Recall@5  Recall@10       MRR      nDCG
0        Base E5  0.323944  0.577465   0.746479  0.447629  0.566178
1  Fine-tuned E5  0.887324  0.985915   0.985915  0.929577  0.946826


In [76]:
comparison_all = pd.DataFrame([
    {
        "Model": "Base E5",
        "Direction": "Sanskrit → English",
        **baseline_sa_en
    },
    {
        "Model": "Fine-tuned E5",
        "Direction": "Sanskrit → English",
        **ft_sa_metrics
    },
    {
        "Model": "Base E5",
        "Direction": "English → Sanskrit",
        **baseline_en_sa
    },
    {
        "Model": "Fine-tuned E5",
        "Direction": "English → Sanskrit",
        **ft_en_metrics
    }
])

print(comparison_all)

           Model           Direction  Recall@1  Recall@5  Recall@10       MRR  \
0        Base E5  Sanskrit → English  0.323944  0.577465   0.746479  0.447629   
1  Fine-tuned E5  Sanskrit → English  0.887324  0.985915   0.985915  0.929577   
2        Base E5  English → Sanskrit  0.788732  0.929577   0.943662  0.847650   
3  Fine-tuned E5  English → Sanskrit  0.887324  0.985915   1.000000  0.933099   

       nDCG  
0  0.566178  
1  0.946826  
2  0.882831  
3  0.949881  


In [77]:
comparison_all.to_csv(
    "model_comparison.csv",
    index=False
)

In [78]:
def retrieve_gita(
    query,
    model,
    documents_df,
    language="english",
    top_k=5
):

    texts = documents_df[language].tolist()

    embeddings = model.encode(
        [
            "passage: " + text
            for text in texts
        ],
        normalize_embeddings=True
    )

    query_embedding = model.encode(
        [
            "query: " + query
        ],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        embeddings
    )[0]

    top_indices = np.argsort(
        scores
    )[::-1][:top_k]

    results = documents_df.iloc[
        top_indices
    ].copy()

    results["similarity"] = scores[
        top_indices
    ]

    return results

# TOP-K RETRIEVAL QUALITY

In [79]:
results = retrieve_gita(
    "What does the Gita say about karma?",
    fine_tuned_model,
    test_df,
    language="english",
    top_k=5
)

print(
    results[
        [
            "verse_id",
            "sanskrit",
            "english",
            "similarity"
        ]
    ]
)

    verse_id  \
33    BG_8_1   
9    BG_2_36   
13    BG_3_1   
63   BG_18_9   
62  BG_17_12   

                                                                                                 sanskrit  \
33  अर्जुन उवाच |किं तद् ब्रह्म किमध्यात्मं किं कर्म पुरुषोत्तम |अधिभूतं च किं प्रोक्तमधिदैवं किमुच्यते |   
9                   अवाच्यवादांश्च बहून्वदिष्यन्ति तवाहिताः |निन्दन्तस्तव सामर्थ्यं ततो दुःखतरं नु किम् |   
13         अर्जुन उवाच |ज्यायसी चेत्कर्मणस्ते मता बुद्धिर्जनार्दन |तत्किं कर्मणि घोरे मां नियोजयसि केशव |   
63            कार्यमित्येव यत्कर्म नियतं क्रियतेऽर्जुन |सङ्गं त्यक्त्वा फलं चैव स त्यागः सात्त्विको मतः |   
62                       अभिसन्धाय तु फलं दम्भार्थमपि चैव यत् |इज्यते भरतश्रेष्ठ तं यज्ञं विद्धि राजसम् |   

                                                                                                                                                                             english  \
33                                          Arjuna said: Krishna

In [80]:
retrieve_gita(
    test_df.iloc[0]["sanskrit"],
    fine_tuned_model,
    test_df,
    language="english",
    top_k=5
)

,chapter,verse,sanskrit,english,transliteration,verse_id,similarity
0,1,11,अयनेषु च सर्वेषु यथाभागमवस्थिताः |भीष्ममेवाभिरक्षन्तु भवन्तः सर्व एव हि |,"Therefore, stationed in your respective positions, all of you should surely protectBhishma on all sides.",ayaneṣu ca sarveṣu yathābhāgamavasthitāḥ .bhīṣmamevābhirakṣantu bhavantaḥ sarva eva hi,BG_1_11,0.632147
7,2,17,अविनाशि तु तद्विद्धि येन सर्वमिदं ततम् |विनाशमव्ययस्यास्य न कश्चित्कर्तुमर्हति |,Know that to be indestructible by which all this is pervaded. No one is able to destroy that imperishable entity.,avināśi tu tadviddhi yena sarvamidaṃ tatam .vināśamavyayasyāsya na kaścitkartumarhati,BG_2_17,0.378954
52,13,30,प्रकृत्यैव च कर्माणि क्रियमाणानि सर्वशः |यः पश्यति तथात्मानमकर्तारं स पश्यति |,"And he alone really sees, who sees all actions being performed in every way by Prakrti alone, and the Self as the non-doer.",prakṛtyaiva ca karmāṇi kriyamāṇāni sarvaśaḥ .yaḥ paśyati tathātmānamakartāraṃ sa paśyati,BG_13_30,0.360905
12,2,70,आपूर्यमाणमचलप्रतिष्ठंसमुद्रमापः प्रविशन्ति यद्वत् |तद्वत्कामा यं प्रविशन्ति सर्वेस शान्तिमाप्नोति न कामकामी |,"As the waters of different rivers enter the ocean, which though full on all sides remains undisturbed, likewise he is whom all enjoyments merge themselves attains peace; not he who hankers after such enjoyments.",āpūryamāṇamacalapratiṣṭhaṃ samudramāpaḥ praviśanti yadvat .tadvatkāmā yaṃ praviśanti sarve sa śāntimāpnoti na kāmakāmī,BG_2_70,0.331981
47,11,28,यथा नदीनां बहवोऽम्बुवेगाःसमुद्रमेवाभिमुखा द्रवन्ति |तथा तवामी नरलोकवीराविशन्ति वक्त्राण्यभिविज्वलन्ति |,"Just as many swift rivers, rushing headlong, enter the ocean, so do these heroes of the human world enter Your blazing mouths. As the myriad streams of rivers rush towards the sea alone, so do those warriors of the mortal world enter Your flaming mouths.",yathā nadīnāṃ bahavo.ambuvegāḥ samudramevābhimukhā dravanti .tathā tavāmī naralokavīrā viśanti vaktrāṇyabhivijvalanti,BG_11_28,0.296308


In [81]:
retrieve_gita(
    test_df.iloc[0]["english"],
    fine_tuned_model,
    test_df,
    language="sanskrit",
    top_k=5
)

,chapter,verse,sanskrit,english,transliteration,verse_id,similarity
0,1,11,अयनेषु च सर्वेषु यथाभागमवस्थिताः |भीष्ममेवाभिरक्षन्तु भवन्तः सर्व एव हि |,"Therefore, stationed in your respective positions, all of you should surely protectBhishma on all sides.",ayaneṣu ca sarveṣu yathābhāgamavasthitāḥ .bhīṣmamevābhirakṣantu bhavantaḥ sarva eva hi,BG_1_11,0.644608
12,2,70,आपूर्यमाणमचलप्रतिष्ठंसमुद्रमापः प्रविशन्ति यद्वत् |तद्वत्कामा यं प्रविशन्ति सर्वेस शान्तिमाप्नोति न कामकामी |,"As the waters of different rivers enter the ocean, which though full on all sides remains undisturbed, likewise he is whom all enjoyments merge themselves attains peace; not he who hankers after such enjoyments.",āpūryamāṇamacalapratiṣṭhaṃ samudramāpaḥ praviśanti yadvat .tadvatkāmā yaṃ praviśanti sarve sa śāntimāpnoti na kāmakāmī,BG_2_70,0.390855
8,2,35,भयाद्रणादुपरतं मंस्यन्ते त्वां महारथाः |येषां च त्वं बहुमतो भूत्वा यास्यसि लाघवम् |,"And the warrior-chiefs who thought highly of you, will now despise you, thinking that it was fear which drove you from battle.",bhayādraṇāduparataṃ maṃsyante tvāṃ mahārathāḥ .yeṣāṃ ca tvaṃ bahumato bhūtvā yāsyasi lāghavam,BG_2_35,0.381620
70,18,46,यतः प्रवृत्तिर्भूतानां येन सर्वमिदं ततम् |स्वकर्मणा तमभ्यर्च्य सिद्धिं विन्दति मानवः |,Man attains the highest perfection by worshipping Him through his own natural duties from whom the tide of creation has streamed forth and by whom all this universe is pervaded.,yataḥ pravṛttirbhūtānāṃ yena sarvamidaṃ tatam .svakarmaṇā tamabhyarcya siddhiṃ vindati mānavaḥ,BG_18_46,0.360792
49,12,3,ये त्वक्षरमनिर्देश्यमव्यक्तं पर्युपासते |सर्वत्रगमचिन्त्यञ्च कूटस्थमचलन्ध्रुवम् |,"Those, however, who fully controlling all their senses and even-minded towards all, and devoted to the welfare of all beings, constantly adore as their very self the unthinkable; omnipresent, indestructible indefinable, eternal, immovable, unmanifest and changeless Brahma, they too come to Me.",ye tvakṣaramanirdeśyamavyaktaṃ paryupāsate .sarvatragamacintyañca kūṭasthamacalandhruvam,BG_12_3,0.357679


In [82]:
translit_queries = []

for _, row in test_df.iterrows():

    translit_queries.append({
        "query": row["transliteration"],
        "target_id": row["verse_id"]
    })

translit_df = pd.DataFrame(
    translit_queries
)

print(translit_df.head())


                                                                                                    query  \
0                  ayaneṣu ca sarveṣu yathābhāgamavasthitāḥ .bhīṣmamevābhirakṣantu bhavantaḥ sarva eva hi   
1  sa ghoṣo dhārtarāṣṭrāṇāṃ hṛdayāni vyadārayat .nabhaśca pṛthivīṃ caiva tumulo.abhyanunādayan (lo vyanu)   
2     hṛṣīkeśaṃ tadā vākyamidamāha mahīpate .arjuna uvāca .senayorubhayormadhye rathaṃ sthāpaya me.acyuta   
3              kathaṃ na jñeyamasmābhiḥ pāpādasmānnivartitum .kulakṣayakṛtaṃ doṣaṃ prapaśyadbhirjanārdana   
4           kulakṣaye praṇaśyanti kuladharmāḥ sanātanāḥ .dharme naṣṭe kulaṃ kṛtsnamadharmo.abhibhavatyuta   

  target_id  
0   BG_1_11  
1   BG_1_19  
2   BG_1_21  
3   BG_1_39  
4   BG_1_40  


In [83]:
translit_embeddings = fine_tuned_model.encode(
    [
        "query: " + q
        for q in translit_df["query"]
    ],
    normalize_embeddings=True
)

In [84]:
translit_scores = cosine_similarity(
    translit_embeddings,
    ft_sanskrit_embeddings
)

translit_rankings = np.argsort(
    translit_scores,
    axis=1
)[:, ::-1]

In [85]:
translit_metrics = calculate_metrics(
    translit_rankings,
    translit_df["target_id"].tolist(),
    test_doc_ids
)

print("Transliteration → Sanskrit")
print(translit_metrics)

Transliteration → Sanskrit
{'Recall@1': 0.07042253521126761, 'Recall@5': 0.2676056338028169, 'Recall@10': 0.323943661971831, 'MRR': np.float64(0.17106923543880928), 'nDCG': np.float64(0.32572642248368344)}


In [86]:
failure_rows = []

for i, ranking in enumerate(ft_sa_rankings):

    predicted_id = test_doc_ids[
        ranking[0]
    ]

    expected_id = test_sa_en.iloc[
        i
    ]["target_id"]

    if predicted_id != expected_id:

        predicted_row = test_df[
            test_df["verse_id"] == predicted_id
        ].iloc[0]

        expected_row = test_df[
            test_df["verse_id"] == expected_id
        ].iloc[0]

        failure_rows.append({
            "query": test_sa_en.iloc[i]["query"],
            "expected_id": expected_id,
            "predicted_id": predicted_id,
            "expected_english": expected_row["english"],
            "predicted_english": predicted_row["english"]
        })

failures_df = pd.DataFrame(
    failure_rows
)

print("Number of failures:", len(failures_df))
print(failures_df.head())

Number of failures: 8
                                                                                      query  \
0  इन्द्रियाणि पराण्याहुरिन्द्रियेभ्यः परं मनः |मनसस्तु परा बुद्धिर्यो बुद्धेः परतस्तु सः |   
1     अजोऽपि सन्नव्ययात्मा भूतानामीश्वरोऽपि सन् |प्रकृतिं स्वामधिष्ठाय सम्भवाम्यात्ममायया |   
2   तद्बुद्धयस्तदात्मानस्तन्निष्ठास्तत्परायणाः |गच्छन्त्यपुनरावृत्तिं ज्ञाननिर्धूतकल्मषाः |   
3      शुचौ देशे प्रतिष्ठाप्य स्थिरमासनमात्मनः |नात्युच्छ्रितं नातिनीचं चैलाजिनकुशोत्तरम् |   
4        आब्रह्मभुवनाल्लोकाः पुनरावर्तिनोऽर्जुन |मामुपेत्य तु कौन्तेय पुनर्जन्म न विद्यते |   

  expected_id predicted_id  \
0     BG_3_42      BG_3_40   
1      BG_4_6     BG_13_33   
2     BG_5_17      BG_3_40   
3     BG_6_11      BG_1_19   
4     BG_8_16       BG_2_9   

                                                                                                                                                                                                                          expected

In [87]:
failures_df.to_csv(
    "retrieval_failures.csv",
    index=False,
    encoding="utf-8"
)

In [88]:
documents = test_df[
    [
        "verse_id",
        "sanskrit",
        "english"
    ]
]

In [89]:
normalize_embeddings=True

# BUILT MINI RAG MODEL

In [90]:
query = "What does the Gita say about karma?"

results = retrieve_gita(
    query,
    fine_tuned_model,
    test_df,
    language="english",
    top_k=3
)

for _, row in results.iterrows():

    print("=" * 60)
    print("Verse:", row["verse_id"])
    print("Sanskrit:", row["sanskrit"])
    print("English:", row["english"])
    print("Score:", row["similarity"])

Verse: BG_8_1
Sanskrit: अर्जुन उवाच |किं तद् ब्रह्म किमध्यात्मं किं कर्म पुरुषोत्तम |अधिभूतं च किं प्रोक्तमधिदैवं किमुच्यते |
English: Arjuna said: Krishna, what is that Brahma, what is Adhyatma, and what is Karma? What is called Adhibhutaand what is termed as Adhidaiva?
Score: 0.5409502983093262
Verse: BG_2_36
Sanskrit: अवाच्यवादांश्च बहून्वदिष्यन्ति तवाहिताः |निन्दन्तस्तव सामर्थ्यं ततो दुःखतरं नु किम् |
English: And your enemies, disparaging your might, will speak many unbecoming words; what can be more distressing than this?
Score: 0.3385365605354309
Verse: BG_3_1
Sanskrit: अर्जुन उवाच |ज्यायसी चेत्कर्मणस्ते मता बुद्धिर्जनार्दन |तत्किं कर्मणि घोरे मां नियोजयसि केशव |
English: Arjuna said: Krishna, if you consider Knowledge as superior to Action, then why do You urge me to this dreadful action, Keshava!
Score: 0.2628054618835449


In [91]:
comparison_all = pd.DataFrame([
    {
        "Model": "Base E5",
        "Direction": "Sanskrit → English",
        **baseline_sa_en
    },
    {
        "Model": "Fine-tuned E5",
        "Direction": "Sanskrit → English",
        **ft_sa_metrics
    },
    {
        "Model": "Base E5",
        "Direction": "English → Sanskrit",
        **baseline_en_sa
    },
    {
        "Model": "Fine-tuned E5",
        "Direction": "English → Sanskrit",
        **ft_en_metrics
    },
    {
        "Model": "Fine-tuned E5",
        "Direction": "Transliteration → Sanskrit",
        **translit_metrics
    }
])

print(comparison_all.to_string(index=False))

comparison_all.to_csv(
    "final_results.csv",
    index=False,
    encoding="utf-8"
)

print("\nSaved to: final_results.csv")

        Model                  Direction  Recall@1  Recall@5  Recall@10      MRR     nDCG
      Base E5         Sanskrit → English  0.323944  0.577465   0.746479 0.447629 0.566178
Fine-tuned E5         Sanskrit → English  0.887324  0.985915   0.985915 0.929577 0.946826
      Base E5         English → Sanskrit  0.788732  0.929577   0.943662 0.847650 0.882831
Fine-tuned E5         English → Sanskrit  0.887324  0.985915   1.000000 0.933099 0.949881
Fine-tuned E5 Transliteration → Sanskrit  0.070423  0.267606   0.323944 0.171069 0.325726

Saved to: final_results.csv
